# Step 2: Embeddings

<!-- notebook-only -->

> **How to use this notebook**
> - **Just reading?** The text cells contain the whole lesson. You can skip every code cell. An even nicer way to read: the website, built with `.venv/bin/python website/build.py --serve`.
> - **Want to practice?** Run the code cells from top to bottom (Shift+Enter). They recompute every table and let you experiment. In **Your turn** cells, replace `None` with your answer: `check(...)` says if it is right and gives a hint if not, and `solution("2.1")` shows the worked answer.
> - Every question also has a **Show the answer** box you can open.

<!-- /notebook-only -->

**Goal:** turn each ID into a short list of numbers (a **vector**), so that tokens with similar meanings get similar numbers.

In [ ]:
# Run this first (practice only): imports and the tokenizer from Step 1
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "toddler").is_dir())
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd

from toddler.bpe import BPETokenizer
from toddler.data import CORPUS
from toddler.exercises import check, solution

np.set_printoptions(formatter={"float_kind": lambda x: f"{x:g}"})
pd.set_option("display.float_format", "{:g}".format)

tok = BPETokenizer.train(CORPUS)
print(len(tok.vocab), "tokens in the vocabulary")

## 2.1 The embedding table

The embedding table has **one row for each token** (53 rows) and **8 columns** (our vector size). Each row is that token's vector. To "embed" a token, you read its row. That's all.

In a real model, the columns have no names and the numbers look random (`0.31, −1.27, 0.08…`). To follow along by hand, we use a cleaned-up version with named columns. It shows what a trained toddler *could* end up with. Below are only the rows we use, grouped by kind; the real table is in ID order (0–52):

| ID | token | me | be | want | stop | feel | food | hug | sleep |
|---|---|---|---|---|---|---|---|---|---|
| 21 | `_I` | 1 | 0 | 0 | 0 | 0 | 0 | 0 | 0 |
| 24 | `_am` | 0 | 1 | 0 | 0 | 0 | 0 | 0 | 0 |
| 28 | `_want` | 0 | 0 | 1 | 0 | 0 | 0 | 0 | 0 |
| 22 | `_.` | 0 | 0 | 0 | 1 | 0 | 0 | 0 | 0 |
| 36 | `_hungry` | 0 | 0 | 0 | 0 | 1 | 1 | 0 | 0 |
| 42 | `_sad` | 0 | 0 | 0 | 0 | 1 | 0 | 1 | 0 |
| 50 | `_tired` | 0 | 0 | 0 | 0 | 1 | 0 | 0 | 1 |
| 39 | `_milk` | 0 | 0 | 0 | 0 | 0 | 1 | 0 | 0 |
| 46 | `_mommy` | 0 | 0 | 0 | 0 | 0 | 0 | 1 | 0 |
| 52 | `_bed` | 0 | 0 | 0 | 0 | 0 | 0 | 0 | 1 |
| 33 | `_hun` (never used) | 0 | 0.38 | −0.31 | −0.09 | −0.71 | −0.41 | 1.38 | 0.52 |
| … | 42 more unused rows | random | | | | | | | |

How to read it:
- The first 4 columns answer "which small word am I?"
- `feel` answers "is it a feeling?" (hungry, sad, tired).
- `food`, `hug` and `sleep` answer "what is it *about*?" A feeling and the thing it wants share the same topic: hungry and milk share `food`, sad and mommy share `hug`, tired and bed share `sleep`.

Embedding `_hungry` means: take its ID (36) and copy row 36.

![Embedding lookup: _hungry → ID 36 → row 36 → its vector](figures/02_embedding_lookup.png)

In [ ]:
# Try it: build the table (random at "birth", then the cleaned-up values in the 10 used rows)
COLUMNS = ["me", "be", "want", "stop", "feel", "food", "hug", "sleep"]

def random_table(vocab_size, dim, seed=0):
    rng = np.random.default_rng(seed)
    return rng.normal(0.0, 0.5, size=(vocab_size, dim)).round(2) + 0.0  # + 0.0 turns -0.0 into 0.0

CLEAN_ROWS = {
    "_I":      [1, 0, 0, 0, 0, 0, 0, 0],
    "_am":     [0, 1, 0, 0, 0, 0, 0, 0],
    "_want":   [0, 0, 1, 0, 0, 0, 0, 0],
    "_.":      [0, 0, 0, 1, 0, 0, 0, 0],
    "_hungry": [0, 0, 0, 0, 1, 1, 0, 0],
    "_sad":    [0, 0, 0, 0, 1, 0, 1, 0],
    "_tired":  [0, 0, 0, 0, 1, 0, 0, 1],
    "_milk":   [0, 0, 0, 0, 0, 1, 0, 0],
    "_mommy":  [0, 0, 0, 0, 0, 0, 1, 0],
    "_bed":    [0, 0, 0, 0, 0, 0, 0, 1],
}

table = random_table(len(tok.vocab), len(COLUMNS))
for token, values in CLEAN_ROWS.items():
    table[tok.token_to_id[token]] = values

def row(token):
    return table[tok.token_to_id[token]]

print("shape:", table.shape, "=", table.size, "numbers")
pd.DataFrame(table, index=tok.vocab, columns=COLUMNS).loc[list(CLEAN_ROWS) + ["_hun"]]

## 2.2 Looking up our sentence, step by step

`I am hungry . I want milk .` → IDs `21 24 36 22 21 28 39 22` → read row 21, then row 24, row 36, row 22, row 21, row 28, row 39, row 22:

| pos | token | me | be | want | stop | feel | food | hug | sleep |
|---|---|---|---|---|---|---|---|---|---|
| 1 | `_I` | 1 | 0 | 0 | 0 | 0 | 0 | 0 | 0 |
| 2 | `_am` | 0 | 1 | 0 | 0 | 0 | 0 | 0 | 0 |
| 3 | `_hungry` | 0 | 0 | 0 | 0 | 1 | 1 | 0 | 0 |
| 4 | `_.` | 0 | 0 | 0 | 1 | 0 | 0 | 0 | 0 |
| 5 | `_I` | 1 | 0 | 0 | 0 | 0 | 0 | 0 | 0 |
| 6 | `_want` | 0 | 0 | 1 | 0 | 0 | 0 | 0 | 0 |
| 7 | `_milk` | 0 | 0 | 0 | 0 | 0 | 1 | 0 | 0 |
| 8 | `_.` | 0 | 0 | 0 | 1 | 0 | 0 | 0 | 0 |

The sentence is now a grid of numbers, and **this grid is what goes into the transformer**. It has 8 rows because the sentence has 8 tokens, and 8 columns because the vector size is 8. The two numbers are the same only by chance.

In [ ]:
# Try it: each ID is a row number, read that row
sentence = "I am hungry . I want milk ."
ids = tok.encode(sentence)
for position, token_id in enumerate(ids, start=1):
    print(f"pos {position}: {tok.vocab[token_id]:<8} → ID {token_id:>2} → row {token_id:>2} = {table[token_id]}")

In [ ]:
# Try it: the grid that goes into the transformer
grid = table[ids]  # NumPy reads all the rows at once
pd.DataFrame(grid, index=[f"{p} {tok.vocab[i]}" for p, i in enumerate(ids, start=1)], columns=COLUMNS)

## 2.3 Measuring "close": the dot product

Multiply two vectors column by column, then add up the results. If they share topics, you get a big number. If they have nothing in common, you get 0.

`_hungry · _sad`:
```
              me    be  want  stop  feel  food   hug sleep
_hungry        0     0     0     0     1     1     0     0
_sad           0     0     0     0     1     0     1     0
multiply       0     0     0     0     1     0     0     0   → sum = 1 (both are feelings)
```

`_hungry · _milk`:
```
              me    be  want  stop  feel  food   hug sleep
_hungry        0     0     0     0     1     1     0     0
_milk          0     0     0     0     0     1     0     0
multiply       0     0     0     0     0     1     0     0   → sum = 1 (same topic: food)
```

`_hungry · _mommy`:
```
              me    be  want  stop  feel  food   hug sleep
_hungry        0     0     0     0     1     1     0     0
_mommy         0     0     0     0     0     0     1     0
multiply       0     0     0     0     0     0     0     0   → sum = 0 (nothing in common)
```

In [ ]:
# Try it: the same three dot products, column by column
def show_dot(a, b):
    va, vb = row(a), row(b)
    print(f"{'':<10}" + "".join(f"{c:>6}" for c in COLUMNS))
    print(f"{a:<10}" + "".join(f"{x:>6g}" for x in va))
    print(f"{b:<10}" + "".join(f"{x:>6g}" for x in vb))
    print(f"{'multiply':<10}" + "".join(f"{x:>6g}" for x in va * vb) + f"   → sum = {va @ vb:g}\n")

show_dot("_hungry", "_sad")
show_dot("_hungry", "_milk")
show_dot("_hungry", "_mommy")

Here are all the pairs, worked out the same way:

| · | hungry | sad | tired | milk | mommy | bed |
|---|---|---|---|---|---|---|
| **hungry** | 2 | 1 | 1 | **1** | 0 | 0 |
| **sad** | 1 | 2 | 1 | 0 | **1** | 0 |
| **tired** | 1 | 1 | 2 | 0 | 0 | **1** |
| **milk** | **1** | 0 | 0 | 1 | 0 | 0 |
| **mommy** | 0 | **1** | 0 | 0 | 1 | 0 |
| **bed** | 0 | 0 | **1** | 0 | 0 | 1 |

The same table as a picture (darker = closer):

![Dot products between tokens](figures/02_similarity.png)

- The 3 feelings are close to each other.
- Each feeling is close to **its own** thing (the bold numbers).
- milk, mommy and bed are *not* close to each other here. They should be, because they all come after "want", but 8 columns are too few to hold everything. That's one reason real models use thousands of columns. Exercise 2.4 fixes it with a 9th column.

Remember the dot product: **attention (Step 4) is built on it.**

In [ ]:
# Try it: every dot product at once (vectors @ vectors.T), drawn as a heatmap
import matplotlib.pyplot as plt

content = ["_hungry", "_sad", "_tired", "_milk", "_mommy", "_bed"]
vectors = np.array([row(t) for t in content])
similarity = vectors @ vectors.T

fig, ax = plt.subplots(figsize=(4.5, 4))
ax.imshow(similarity, cmap="Blues")
ax.set_xticks(range(len(content)), content, rotation=45)
ax.set_yticks(range(len(content)), content)
for i in range(len(content)):
    for j in range(len(content)):
        ax.text(j, i, f"{similarity[i, j]:g}", ha="center", va="center")
ax.set_title("dot products")
plt.show()

## 2.4 Bonus: you can add and subtract meaning

Work column by column, like a sum on paper:

```
              me    be  want  stop  feel  food   hug sleep
  _milk        0     0     0     0     0     1     0     0
− _hungry      0     0     0     0     1     1     0     0
+ _sad         0     0     0     0     1     0     1     0
= result       0     0     0     0     0     0     1     0   → _mommy
```

The `feel` column is 0 − 1 + 1 = 0, the `food` column is 1 − 1 + 0 = 0, and the `hug` column is 0 − 0 + 1 = 1. In words: milk is to hungry what mommy is to sad. Real embeddings show the same effect, roughly: king − man + woman ≈ queen.

In [ ]:
# Try it: milk - hungry + sad, then find the token closest to the result (smallest distance)
result = row("_milk") - row("_hungry") + row("_sad")

print(f"{'':<10}" + "".join(f"{c:>6}" for c in COLUMNS))
for label, vector in [("  _milk", row("_milk")), ("- _hungry", row("_hungry")), ("+ _sad", row("_sad")), ("= result", result)]:
    print(f"{label:<10}" + "".join(f"{x:>6g}" for x in vector))

closest = min(CLEAN_ROWS, key=lambda t: np.linalg.norm(result - row(t)))
print("\nclosest token:", closest)

## 2.5 Where do the numbers come from?

Nobody writes them. At "birth", every row is random. Here is `_hungry` as an example:

```
                  me    be  want  stop  feel  food   hug sleep
at birth       -0.32  0.37 -0.59 -0.72  0.32  0.38 -0.48  0.28
after training     0     0     0     0     1     1     0     0   (cleaned up)
```

Training (Step 7) changes the rows a little at a time. Tokens used in similar places end up with similar rows. hungry, sad and tired all come after `_am` and before `_.`, so they end up sharing a direction (our `feel` column).

The embedding table is not trained on its own, before the rest: every table of the model is trained at the same time, in the same rounds. Step 3 (section 3.8) shows why there is no chicken-and-egg problem.

The 43 unused rows (`_hun`, `h`, …) never show up in the training text. Training never touches them, so they stay random, and their dot products mean nothing. In real models, tokens that were rarely seen in training cause strange "glitch token" behavior.

In [ ]:
# Try it: _hungry at birth vs after training, and an unused row
at_birth = random_table(len(tok.vocab), len(COLUMNS))
print("_hungry at birth      :", at_birth[tok.token_to_id["_hungry"]])
print("_hungry after training:", row("_hungry"))
print("\n_hun (never trained)  :", row("_hun"))
for t in ["_hungry", "_milk", "_I"]:
    print(f"_hun · {t:<8} = {row('_hun') @ row(t):+.2f}   (meaningless)")

## 2.6 Why not just use "one-hot" vectors?

A one-hot vector is 53 zeros with a single 1 at the token's ID. It names the token, but:
- The dot product of any two different one-hot vectors is 0. So hungry is exactly as far from sad as from bed; nothing is ever "close".
- It is as long as the vocabulary (50,000+ numbers in GPT).

Reading row 36 of the table gives the same result as multiplying the one-hot vector by the table:

```
one-hot(_hungry) = 0 0 0 … 0 1 0 … 0        (53 numbers, the only 1 is at position 36)
one-hot × table  = 0·row0 + 0·row1 + … + 1·row36 + … + 0·row52 = row 36
```

So the embedding table is just an ordinary weight matrix, and training adjusts it like every other weight.

In [ ]:
# Try it: one-hot × table gives the same row as a lookup
one_hot = np.zeros(len(tok.vocab))
one_hot[tok.token_to_id["_hungry"]] = 1
print("one-hot(_hungry):", one_hot.astype(int))
print("one-hot × table :", one_hot @ table)
print("row 36          :", table[36])
print("same?           ", np.array_equal(one_hot @ table, table[36]))

## 2.7 Real scale

| | Toddler | GPT-3 |
|---|---|---|
| Rows (vocabulary) | 53 | 50,257 |
| Columns (vector size) | 8 | 12,288 |
| Numbers in the table | 424 | ~618 million |

## 2.8 What is still missing

Look again at the grid in 2.2:
- Rows 1 and 5 (`_I`) are **exactly the same**. But the first `_I` is followed by `_am` and the second by `_want`, and the model can't tell them apart yet. This is solved in **Step 3: positions**.
- Row 6 (`_want`) only knows "want". It has no idea the child said *hungry*, so it can't know that milk comes next. This is solved in **Step 4: attention**.

In [ ]:
# Try it: rows 1 and 5 of the grid
print("pos 1:", grid[0])
print("pos 5:", grid[4])
print("identical?", np.array_equal(grid[0], grid[4]))

<!-- notebook-only -->

The same code is saved in `toddler/embeddings.py`, and the next notebooks import it from there.

<!-- /notebook-only -->

In [ ]:
# Try it: the package version used by the next notebooks builds the same table
from toddler import embeddings

assert np.array_equal(embeddings.toddler_table(tok), table)
print("toddler/embeddings.py builds the same table")

## Your turn

### Your turn 2.1: compute `_tired · _bed` column by column.

<details>
<summary>Show the answer</summary>
<!-- answer: 1 | one -->
<!-- format: one number -->

```
              me    be  want  stop  feel  food   hug sleep
_tired         0     0     0     0     1     0     0     1
_bed           0     0     0     0     0     0     0     1
multiply       0     0     0     0     0     0     0     1   → sum = 1
```

The dot product is the **sum** of the multiply row: **1**, a single number (they share `sleep`).

</details>

In [ ]:
my_answer = None  # one number
check("2.1", my_answer)

### Your turn 2.2: compute `_bed − _tired + _hungry`. Which token do you get?

<details>
<summary>Show the answer</summary>
<!-- answer: _milk | milk -->
<!-- format: a token, e.g. _sad -->

```
              me    be  want  stop  feel  food   hug sleep
  _bed         0     0     0     0     0     0     0     1
− _tired       0     0     0     0     1     0     0     1
+ _hungry      0     0     0     0     1     1     0     0
= result       0     0     0     0     0     1     0     0   → _milk
```

Bed is to tired what milk is to hungry.

</details>

In [ ]:
my_answer = None  # a token, e.g. "_sad"
check("2.2", my_answer)

<!-- notebook-only -->

### Your turn 2.3 (code): write the dot product yourself, with a loop (no `@`, no `np.dot`).

<details>
<summary>Show the answer</summary>

```python
def my_dot(a, b):
    total = 0
    for i in range(len(a)):
        total += a[i] * b[i]
    return total
```

</details>

<!-- /notebook-only -->

In [ ]:
def my_dot(a, b):
    total = 0
    # your code here
    return None

check("2.3", my_dot)

### Your turn 2.4: add a 9th column, `thing`, so that milk, mommy and bed become close to each other.

Which rows get a 1 in the `thing` column? What do the dot products look like then?

<details>
<summary>Show the answer</summary>
<!-- choices: *The rows of _milk, _mommy and _bed | The rows of _hungry, _sad and _tired | All 10 rows our sentences use | Only the row of _milk -->

Put a 1 in the `thing` column of `_milk`, `_mommy` and `_bed`, and 0 everywhere else. The dot products become:

| · | hungry | sad | tired | milk | mommy | bed |
|---|---|---|---|---|---|---|
| **hungry** | 2 | 1 | 1 | 1 | 0 | 0 |
| **sad** | 1 | 2 | 1 | 0 | 1 | 0 |
| **tired** | 1 | 1 | 2 | 0 | 0 | 1 |
| **milk** | 1 | 0 | 0 | 2 | **1** | **1** |
| **mommy** | 0 | 1 | 0 | **1** | 2 | **1** |
| **bed** | 0 | 0 | 1 | **1** | **1** | 2 |

The things are now close to each other, like the feelings, and nothing else changed. One more column stores one more kind of similarity.

</details>

In [ ]:
table9 = np.hstack([table, np.zeros((len(tok.vocab), 1))])
# your code here (tip: tok.token_to_id["_milk"] is the row of _milk; column 8 is the new one)

check("2.4", table9)

In [ ]:
# Try it: the new dot products (run after 2.4)
vectors9 = np.array([table9[tok.token_to_id[t]] for t in content])
pd.DataFrame(vectors9 @ vectors9.T, index=content, columns=content)

### Your turn 2.5 (think): rows 1 and 5 of our grid are exactly the same. Why is that a problem for guessing the next token?

<details>
<summary>Show the answer</summary>
<!-- choices: *Same input, same output: but the right next tokens differ (_am, then _want) | The model would crash on a repeated token | Storing the same row twice wastes memory | It is not a problem at all -->

Same input, same output: the model would make the same guess at both places. But the right answers are different: `_am` after the first `_I`, `_want` after the second. The model needs to know *where* each token is (Step 3) and *what came before* (Step 4).

</details>

## Summary

- The embedding table has one row per token. Embedding = reading the row of each ID.
- The numbers start random and are learned. Tokens used in similar ways end up with similar rows.
- The **dot product** measures how close two vectors are. Attention is built on it.
- Meaning can be added and subtracted: milk − hungry + sad = mommy.
- Each row only knows its own token: no position, no context. Next: **Step 3, positions**.